In [2]:
!pip install -q transformers datasets pillow gradio evaluate
!pip install -q torchvision --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 917.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.

In [4]:
from google.colab import drive, userdata
from huggingface_hub import login
import torch
from PIL import Image
import numpy as np

drive.mount('/content/drive')

token = userdata.get('HF_TOKEN')
login(token=token)
print("Login successful!")

Mounted at /content/drive
Login successful!


# STEP 1 — Fine-tune Model

## Part 1: Import and Load Model

In [5]:
from transformers import ViTForImageClassification, ViTImageProcessor

label2id = {"cat": 0, "dog": 1}
id2label = {0: "cat", 1: "dog"}

MODEL_NAME = "google/vit-base-patch16-224"

processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("Model ready!")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Model ready!


In [6]:
# CELL 4B — Debug: Labels check karo
print("Sample check:")
for i in range(min(5, len(full_dataset))):
    sample = full_dataset[i]
    print(f"  Index {i} → label: {sample['labels']}, pixels shape: {sample['pixel_values'].shape}")

print(f"\nUnique labels in dataset:")
all_labels = [full_dataset[i]['labels'].item() for i in range(len(full_dataset))]
print(f"  {set(all_labels)}")
print(f"  Expected: {{0, 1}}")

Sample check:


NameError: name 'full_dataset' is not defined

In [22]:
# CELL 4C — Folders check karo
import os

DATA_DIR = "/content/drive/MyDrive/cat_dog_data/train"

print("Folders found:")
for i, name in enumerate(sorted(os.listdir(DATA_DIR))):
    full_path = os.path.join(DATA_DIR, name)
    is_dir = os.path.isdir(full_path)
    print(f"  index {i} → '{name}' (folder: {is_dir})")

Folders found:
  index 0 → 'cat' (folder: True)
  index 1 → 'dog' (folder: True)


In [18]:
# CELL 4D — .DS_Store delete karo
import os

ds_store = "/content/drive/MyDrive/cat_dog_data/train/.DS_Store"

if os.path.exists(ds_store):
    os.remove(ds_store)
    print(".DS_Store delete ho gaya!")
else:
    print("File nahi mili — already clean hai")

# Verify karo
print("\nFolders ab:")
for i, name in enumerate(sorted(os.listdir(
        "/content/drive/MyDrive/cat_dog_data/train"))):
    print(f"  index {i} → '{name}'")

.DS_Store delete ho gaya!

Folders ab:
  index 0 → 'cat'
  index 1 → 'dog'


In [27]:
# CELL 4E — Force reload
import importlib, os, torch
from PIL import Image

# Purana dataset delete karo memory se
try:
    del full_dataset, train_ds, val_ds
    print("Old dataset cleared!")
except:
    print("Nothing to clear")

# Verify folders clean hain
DATA_DIR = "/content/drive/MyDrive/cat_dog_data/train"
print("\nFolders right now:")
all_items = sorted(os.listdir(DATA_DIR))
for i, name in enumerate(all_items):
    print(f"  {i} → '{name}'")

Old dataset cleared!

Folders right now:
  0 → 'cat'
  1 → 'dog'


In [28]:
# CELL 4F — Fresh dataset load
from torch.utils.data import Dataset, DataLoader, random_split

class CatDogDataset(Dataset):
    def __init__(self, root_dir, processor):
        self.processor = processor
        self.samples   = []

        # Sirf actual folders lo — hidden files ignore karo
        classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
            and not d.startswith('.')    # ← .DS_Store aur sab hidden files ignore
        ])

        self.class_to_idx = {c: i for i, c in enumerate(classes)}

        for class_name in classes:
            class_dir = os.path.join(root_dir, class_name)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.jpg','.jpeg','.png','.webp')):
                    self.samples.append((
                        os.path.join(class_dir, fname),
                        self.class_to_idx[class_name]
                    ))

        print(f"Classes : {self.class_to_idx}")
        print(f"Total   : {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": torch.tensor(label)
        }

# Fresh load
full_dataset = CatDogDataset(DATA_DIR, processor)

# Split
val_size   = max(1, int(0.2 * len(full_dataset)))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train : {train_size}")
print(f"Val   : {val_size}")

# Verify labels
all_labels = [full_dataset[i]['labels'].item() for i in range(len(full_dataset))]
print(f"Labels: {set(all_labels)}  ← should be {{0, 1}}")

Classes : {'cat': 0, 'dog': 1}
Total   : 31
Train : 25
Val   : 6
Labels: {0, 1}  ← should be {0, 1}


## Part B — Data Preprocessing

In [7]:
import os
from torch.utils.data import Dataset, DataLoader, random_split

class CatDogDataset(Dataset):
    """
    Custom dataset — directly PIL images load karta hai
    torchvision ka use nahi karta
    """
    def __init__(self, root_dir, processor):
        self.processor = processor
        self.samples   = []   # (image_path, label) pairs

        classes = sorted(os.listdir(root_dir))   # ['cat', 'dog']
        self.class_to_idx = {c: i for i, c in enumerate(classes)}

        for class_name in classes:
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    self.samples.append((
                        os.path.join(class_dir, fname),
                        self.class_to_idx[class_name]
                    ))

        print(f"Classes found : {self.class_to_idx}")
        print(f"Total images  : {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)  # (3, 224, 224)
        return {"pixel_values": pixel_values, "labels": torch.tensor(label)}


# Load dataset
DATA_DIR = "/content/drive/MyDrive/cat_dog_data/train"

full_dataset = CatDogDataset(DATA_DIR, processor)

# 80% train, 20% validation split
val_size   = max(1, int(0.2 * len(full_dataset)))
train_size = len(full_dataset) - val_size

train_ds, val_ds = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"\nTraining images  : {train_size}")
print(f"Validation images: {val_size}")

Classes found : {'cat': 0, 'dog': 1}
Total images  : 31

Training images  : 25
Validation images: 6


## Part C — Training

In [9]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

metric = evaluate.load("accuracy")

def compute_metrics(pred):
    predictions = np.argmax(pred.predictions, axis=1)
    return metric.compute(
        predictions=predictions,
        references=pred.label_ids
    )

# ← Apna HF username yahan daalo
HF_USERNAME = "VishRoy"

training_args = TrainingArguments(
    output_dir="cat-dog-model",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=5,
    push_to_hub=True,
    hub_model_id=f"{HF_USERNAME}/cat-dog-classifier",
    report_to="none",
    remove_unused_columns=False    # ← Important for custom dataset
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print("Training shuru...")
trainer.train()
print("Training complete!")

Training shuru...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.000322,0.001748,1.000000
2,0.000054,0.000663,1.000000
3,0.000006,0.000360,1.000000
4,0.000004,0.000252,1.000000
5,0.000003,0.000204,1.000000
6,0.000003,0.000180,1.000000
7,0.000002,0.000168,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

Training complete!


In [10]:
trainer.push_to_hub()
processor.push_to_hub(f"{HF_USERNAME}/cat-dog-classifier")

print(f"\nModel live hai!")
print(f"https://huggingface.co/{HF_USERNAME}/cat-dog-classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...g-model/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...g-model/model.safetensors:  44%|####4     |  152MB /  343MB            

README.md:   0%|          | 0.00/2.06k [00:00<?, ?B/s]


Model live hai!
https://huggingface.co/VishRoy/cat-dog-classifier


# STEP 2 — Test your Model

In [12]:
def predict(image_path):
    image  = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        probs = torch.softmax(
            model(**inputs).logits, dim=-1
        )[0]

    cat_pct = float(probs[0]) * 100
    dog_pct = float(probs[1]) * 100

    winner = "🐱 CAT" if cat_pct > dog_pct else "🐶 DOG"
    print(f"Result : {winner}")
    print(f"Cat    : {cat_pct:.1f}%")
    print(f"Dog    : {dog_pct:.1f}%")

In [15]:
import gradio as gr

def classify_image(image):
    inputs = processor(images=image.convert("RGB"), return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    return {
        "🐱 Cat": float(probs[0]),
        "🐶 Dog": float(probs[1])
    }

demo = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil", label="Upload Photo"),
    outputs=gr.Label(num_top_classes=2, label="Result"),
    title="🐱🐶 Cat vs Dog Classifier",
    description="Description of Photo",
    theme="soft"
)

demo.launch(share=True)
!gradio deploy

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c85d3d0444e3ead9af.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Need 'write' access token to create a Spaces repo.
Creating new Spaces Repo in '/content'. Collecting metadata, press Enter to 
accept default value.
Enter Spaces app title [content]: cat-dog
Enter Gradio app file : cat-dog
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /usr/local/lib/python3.12/dist-packages/gradio/cli/commands/deploy_space.py: │
│ 301 in deploy                                                                │
│                                                                              │
│   298 │   │   print(                                                         │
│   299 │   │   │   f"Creating new Spaces Repo in '{repo_directory}'.          │
│       Collecting metadata, press Enter to accept default value."             │
│   300 │   │   )                                                              │
│ ❱ 301 │   │   configuration = add_configuration_to_readme(                   │
│   302 │   │   │   title,                     

In [24]:
# app.py

app_code = """
import gradio as gr
import torch
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor

MODEL_ID = "VishRoy/cat-dog-classifier"
processor = ViTImageProcessor.from_pretrained(MODEL_ID)
model = ViTForImageClassification.from_pretrained(MODEL_ID)
model.eval()

def classify_image(image):
    inputs = processor(
        images=image.convert("RGB"),
        return_tensors="pt"
    )
    with torch.no_grad():
        probs = torch.softmax(
            model(**inputs).logits, dim=-1
        )[0]
    return {
        "Cat": float(probs[0]),
        "Dog": float(probs[1])
    }

demo = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil", label="Upload Photo"),
    outputs=gr.Label(num_top_classes=2, label="Result"),
    title="Cat vs Dog Classifier",
    description="Photo Description",
    theme="soft"
)

demo.launch()
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py fixed!")

app.py fixed!


In [25]:
# CELL — requirements.txt
req = """transformers
torch
Pillow
gradio
"""

with open("requirements.txt", "w") as f:
    f.write(req)

print("requirements.txt ready!")

requirements.txt ready!


In [26]:
# CELL — Space pe push karo
from huggingface_hub import HfApi

api = HfApi()

# Space banao (agar pehle se nahi hai)
try:
    api.create_repo(
        repo_id="VishRoy/cat-dog-app",
        repo_type="space",
        space_sdk="gradio",
        exist_ok=True
    )
    print("Space ready!")
except Exception as e:
    print(f"Space already exists: {e}")

# Files upload karo
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id="VishRoy/cat-dog-app",
    repo_type="space"
)

api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id="VishRoy/cat-dog-app",
    repo_type="space"
)

print("Deploy ho gaya!")
print("URL: https://huggingface.co/spaces/VishRoy/cat-dog-app")

Space ready!


No files have been modified since last commit. Skipping to prevent empty commit.


Deploy ho gaya!
URL: https://huggingface.co/spaces/VishRoy/cat-dog-app


In [28]:
# Ek zip mein sab pack karo
import zipfile

with zipfile.ZipFile("cat-dog-project.zip", "w") as zf:
    zf.write("app.py")
    zf.write("requirements.txt")

print("cat-dog-project.zip ready!")
print("Left sidebar se download karo")

cat-dog-project.zip ready!
Left sidebar se download karo
